# 03 — Population: from age-detail files to a municipality-year panel

**Goal.** Transform the POSAS population releases into a compact annual municipality panel that can later be combined with OMI quotations and transaction volumes.

The key methodological choice is simple: the raw municipality file contains one record for each age plus an official **`Età = 999` total row**. For the market panel we use that official total rather than summing ages ourselves.

**Source:** ISTAT — POSAS population by municipality.

## 1. Setup and source inventory

Population files are much larger than the final panel because they contain age detail. We therefore discover the available years first and load only the columns needed for the municipality total.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns',40)
pd.set_option('display.float_format',lambda x:f'{x:,.2f}')

def find_project_root(start=None):
    start=Path(start or Path.cwd()).resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'data'/'raw'/'population').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/population.')

PROJECT_ROOT=find_project_root()
RAW_DIR=PROJECT_ROOT/'data'/'raw'/'population'
years=sorted(int(p.name) for p in RAW_DIR.iterdir() if p.is_dir() and p.name.isdigit() and (p/f'POSAS_{p.name}_it_Comuni.csv').exists())

print(f'Population releases: {years}')
print(f'Number of releases: {len(years):,}')

## 2. Extract the official municipality total

Reading only six columns keeps the operation lightweight. `Età = 999` is selected **before** aggregation because it is already the official municipality total. This avoids double-counting or introducing rounding differences by reconstructing totals from age classes.

In [ ]:
def load_municipality_totals(year):
    path=RAW_DIR/str(year)/f'POSAS_{year}_it_Comuni.csv'
    df=pd.read_csv(path,sep=';',encoding='utf-8-sig',usecols=['Codice comune','Comune','Età','Totale maschi','Totale femmine','Totale'])
    df.columns=[str(c).replace('\ufeff','').strip() for c in df.columns]
    df['Età']=pd.to_numeric(df['Età'],errors='coerce')
    totals=df.loc[df['Età'].eq(999)].copy()

    totals=totals.rename(columns={'Codice comune':'municipality_code','Totale':'population'})
    totals['municipality_code']=totals['municipality_code'].astype('string').str.extract(r'(\d+)')[0].str.zfill(6)
    totals['population']=pd.to_numeric(totals['population'],errors='coerce')
    totals['year']=year

    if totals['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate municipality total keys')
    if totals['population'].isna().any():
        raise ValueError(f'{year}: missing population totals')

    return totals[['municipality_code','Comune','population','year']]

population=pd.concat([load_municipality_totals(year) for year in years],ignore_index=True)

print(f'Rows in population panel: {len(population):,}')
print(f'Unique municipality codes: {population.municipality_code.nunique():,}')
print(f'Years: {population.year.min()}–{population.year.max()}')
display(population.head())

## 3. Validate the panel structure

The intended grain is **one row per municipality and year**. This is an important control because later joins use exactly this key. Municipality names are retained for readability, but the code is the identifier used for uniqueness checks.

In [ ]:
duplicates=population.duplicated(['year','municipality_code']).sum()
missing_codes=population.municipality_code.isna().sum()
missing_population=population.population.isna().sum()

print(f'Duplicate year + municipality keys: {duplicates:,}')
print(f'Missing municipality codes: {missing_codes:,}')
print(f'Missing population values: {missing_population:,}')

if duplicates or missing_codes or missing_population:
    raise ValueError('Population panel failed the structural validation.')

year_control=(population.groupby('year',as_index=False).agg(municipalities=('municipality_code','nunique'),population=('population','sum')).sort_values('year'))
display(year_control)

## 4. National population dynamics

The annual national total is obtained by summing municipality totals. Year-on-year growth is calculated only after the data have been reduced to one national observation per year.

In [ ]:
national_population=(population.groupby('year',as_index=False).agg(population=('population','sum'),municipalities=('municipality_code','nunique')).sort_values('year'))
national_population['population_yoy_pct']=national_population['population'].pct_change().mul(100)
display(national_population)

fig,ax=plt.subplots(figsize=(11,5))
ax.plot(national_population['year'],national_population['population'],marker='o')
ax.set(title='Italian resident population — municipality totals',xlabel='Year',ylabel='Residents')
ax.grid(alpha=.25)
plt.show()

## 5. Municipality growth and distribution

Growth rates are calculated within municipality, comparing each year with the previous available year. This is different from a national growth rate and must not be computed on the raw age-detail rows.

The example below also shows how to identify large and small municipalities without changing the underlying panel.

In [ ]:
population=population.sort_values(['municipality_code','year']).reset_index(drop=True)
population['population_yoy_pct']=population.groupby('municipality_code')['population'].pct_change().mul(100)

latest_year=population.year.max()
latest_population=population.loc[population.year.eq(latest_year)].copy()

print(f'Latest year: {latest_year}')
display(latest_population.nlargest(20,'population')[['municipality_code','Comune','population']])

display(population.loc[population.year.eq(latest_year),'population'].describe().to_frame('population'))

## 6. Geography and interpretation

Population is a **stock** measured annually. It is therefore not directly comparable with a semester-level OMI quotation or an annual NTN flow without respecting the different time frequencies.

For later integration, the population panel is deliberately kept at the clean grain `year + municipality_code`. The municipality name is descriptive metadata, not the join key.

In [ ]:
# Compact analytical output used by downstream notebooks.
population_panel=population[['year','municipality_code','Comune','population','population_yoy_pct']].copy()

assert not population_panel.duplicated(['year','municipality_code']).any()

print('Final population panel:')
print(f'  rows: {len(population_panel):,}')
print(f'  municipalities: {population_panel.municipality_code.nunique():,}')
print(f'  years: {population_panel.year.nunique():,}')

display(population_panel.head())

## 7. What this notebook establishes

- The official `Età = 999` record is used as the municipality total.
- The analytical grain is explicitly **municipality × year**.
- Population growth is calculated after the correct aggregation, not on age-detail observations.
- Municipality names are preserved for interpretation, while the six-digit ISTAT code is the structural key.
- The population panel is annual, so any future comparison with OMI must respect OMI's semester frequency.

This separation of concepts is intentional: the integration notebook should combine **price quotations, transaction activity and population** only after each source has passed its own quality controls.